# 🧠 Build Your Own AI Brain: RAG & Contextual Embeddings

Welcome to the workshop! In this notebook, we are going to build two things:
1. **A RAG (Retrieval-Augmented Generation) System:** An AI that doesn't hallucinate because it reads from a custom textbook (Pinecone).
2. **Contextual Embeddings:** Proof that modern AI understands the *meaning* of a word based on its surroundings, not just the spelling.

---

### 🛠️ Step 1: Install Dependencies & Setup Keys
We only need two libraries: Google's AI SDK and Pinecone's database SDK.

In [ ]:
!pip install google-generativeai pinecone-client numpy --quiet

In [ ]:
import os
import time
import numpy as np
import google.generativeai as genai
from pinecone import Pinecone, ServerlessSpec

# 🔑 PASTE YOUR GEMINI API KEY HERE (Get free from aistudio.google.com)
GEMINI_API_KEY = 'PASTE_YOUR_GEMINI_KEY_HERE' 

# 🔑 PASTE YOUR PINECONE API KEY HERE (Get free from pinecone.io)
PINECONE_API_KEY = 'PASTE_YOUR_PINECONE_KEY_HERE' 

# Configure APIs
genai.configure(api_key=GEMINI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print('✅ Libraries imported and APIs configured!')

---
### 📚 Part 1: Building the RAG System

#### 1. The Knowledge Base
AI doesn't know about our specific college. We have to give it a memory. We are going to feed it 3 facts about a fictional 'Tech Fest 2026'.

In [ ]:
documents = [
    {'id': 'fact1', 'text': 'The College Tech Fest 2026 will be held on October 15th. The main event is the 24-hour Hackathon.'},
    {'id': 'fact2', 'text': 'Registration for the Tech Fest is completely free for all Second Year (SY) and Third Year (TY) students. First Year students need faculty permission.'},
    {'id': 'fact3', 'text': 'The prize pool for the Hackathon is $5000. The first-place team also receives high-end mechanical keyboards.'}
]

print(f'✅ Loaded {len(documents)} facts into memory.')

#### 2. Create the Vector Database (Pinecone)
Pinecone is our AI's long-term memory. We are creating a specific 'brain' for the Tech Fest. *(Note: Creating the index takes about 60 seconds).* 

In [ ]:
INDEX_NAME = 'tech-fest-rag-brain'

if INDEX_NAME not in pc.list_indexes().names():
    print('🧠 Creating Pinecone Index (Please wait ~60 seconds)...')
    pc.create_index(
        name=INDEX_NAME,
        dimension=768, # Gemini's embedding model outputs 768 dimensions
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    time.sleep(60) 
    print('✅ Index created and ready!')
else:
    print('✅ Index already exists!')

index = pc.Index(INDEX_NAME)

#### 3. Embed and Store the Facts (The 'Learning' Phase)
Computers don't understand text; they understand numbers. We use Gemini to turn our text into numbers (Embeddings) and store them in Pinecone.

In [ ]:
EMBEDDING_MODEL = 'models/text-embedding-004'
vectors_to_upsert = []

for doc in documents:
    # 1. Convert text to numbers (Embedding)
    embedding_response = genai.embed_content(
        model=EMBEDDING_MODEL, 
        content=doc['text']
    )
    
    # 2. Prepare the data for Pinecone
    vectors_to_upsert.append({
        'id': doc['id'],
        'values': embedding_response['embedding'],
        'metadata': {'text': doc['text']} # Store the original text to read later
    })

# 3. Upload to Pinecone
index.upsert(vectors=vectors_to_upsert)
print(f'✅ Successfully stored {len(vectors_to_upsert)} facts in the AI\'s memory!')

#### 4. The RAG Query (The 'Magic' Phase)
Now we ask a question. The AI will search its memory for the closest matching facts, and then use Gemini to write a human-readable answer.

In [ ]:
# 1. The User's Question
user_question = 'What is the prize for the hackathon and when is the fest?'
print(f'❓ User asked: {user_question}\n')

# 2. Convert the question into numbers (Embedding)
query_embedding = genai.embed_content(
    model=EMBEDDING_MODEL, 
    content=user_question
)['embedding']

# 3. Search Pinecone for the most relevant facts
search_results = index.query(
    vector=query_embedding,
    top_k=2, # Get the top 2 most relevant facts
    include_metadata=True
)

# Extract the text from the search results
retrieved_context = '\n'.join([match['metadata']['text'] for match in search_results['matches']])
print(f'🔍 AI found these facts in memory:\n{retrieved_context}\n')

# 4. Augment: Combine the facts and the question into a prompt
rag_prompt = f'''You are a helpful college assistant. 
Answer the user's question using ONLY the provided context. 
If the answer is not in the context, say "I don't have that information."

Context: 
{retrieved_context}

Question: 
{user_question}
'''

# 5. Generate: Ask Gemini to write the final answer
gemini_model = genai.GenerativeModel('gemini-1.5-flash')
response = gemini_model.generate_content(rag_prompt)

print('🤖 AI Answer:')
print('-' * 40)
print(response.text)

---
### 🪄 Part 2: The 'Context' Magic (Contextual Embeddings)

Older AI gave the word 'bank' the exact same mathematical ID whether you meant a river or money. Modern AI (Transformers) changes the numbers based on the **context** of the sentence. Let's prove it.

In [ ]:
# Helper function to calculate Cosine Similarity (how close two vectors are)
# 1.0 = Identical meaning, 0.0 = Completely unrelated
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print('🧠 Testing how AI understands the word "BANK" in different contexts...\n')

# We embed short phrases, because context matters!
phrases = {
    'Financial Bank 1': 'I went to the bank to deposit my salary.',
    'Financial Bank 2': 'The financial bank approved my home loan.',
    'River Bank 1': 'We had a picnic on the muddy river bank.',
    'River Bank 2': 'The water level of the river bank rose after the rain.'
}

# 1. Get embeddings for all phrases
embeddings = {}
for name, text in phrases.items():
    response = genai.embed_content(model=EMBEDDING_MODEL, content=text)
    embeddings[name] = response['embedding']

# 2. Compare them!
print('📊 COSINE SIMILARITY SCORES (1.0 = Perfect Match, 0.0 = Unrelated):\n')

# Compare Financial vs Financial
sim_fin_fin = cosine_similarity(embeddings['Financial Bank 1'], embeddings['Financial Bank 2'])
print(f'💰 Financial vs Financial: {sim_fin_fin:.4f}  <-- (High similarity, different sentences!)')

# Compare River vs River
sim_riv_riv = cosine_similarity(embeddings['River Bank 1'], embeddings['River Bank 2'])
print(f'🌊 River vs River:       {sim_riv_riv:.4f}  <-- (High similarity, different sentences!)')

# Compare Financial vs River (The Magic Moment)
sim_fin_riv = cosine_similarity(embeddings['Financial Bank 1'], embeddings['River Bank 1'])
print(f'💰 Financial vs River:   {sim_fin_riv:.4f}  <-- (Low similarity, even though both use the word "BANK"!)')

print('\n✨ CONCLUSION: The AI doesn\'t just look at the word "bank". It looks at "deposit", "loan", "muddy", and "river" to completely change the mathematical meaning of the word!')